"""
================================================================================
  Understanding Energy Use Dynamics in Indian Farms: Efficiency and Sustainability
  Author  : Dipankar Yadav (Roll No. 21AG36011)
  Guide   : Prof. Ankit Shekhar & Prof. Prithwiraj Dey
  Dept.   : Agricultural and Food Engineering, IIT Kharagpur
================================================================================
"""

# ============================================================
# CELL 1 — INSTALL & IMPORT DEPENDENCIES
# ============================================================

In [ ]:
import re
import random
import warnings
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import openpyxl
from openpyxl.drawing.image import Image as XLImage
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from scipy.optimize import linprog
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

warnings.filterwarnings("ignore")
matplotlib.use("Agg")
print("All libraries imported successfully.")

All libraries imported successfully.


In [ ]:
import os
from google.colab import drive

# Unmount and clear any existing cache
drive.flush_and_unmount()

# Ensure the mount point directory is empty
if os.path.exists("/content/drive"):
    os.system("rm -rf /content/drive")

# Mount the drive again
drive.mount("/content/drive", force_remount=True)

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


# ============================================================
# CELL 2 — CONFIGURATION
# ============================================================

In [ ]:
# ── Select the researcher whose data will be processed ──────
PERSON_NAME = "Akhil"

FILE_PATHS = {
    "Vinit":    "VP_Data.xlsx",
    "Abhay":    "APS_Data.xlsx",
    "Toko":     "Toko_Data.xlsx",
    "Dipankar": "DIP_Data.xlsx",
    "Akhil":    "MA_Data.xlsx",
    # "Rudra":    "RPSC_Data.xlsx",
}

STATE_MAP = {
    "Vinit":    "Madhya Pradesh",
    "Abhay":    "Uttar Pradesh",
    "Dipankar": "Uttar Pradesh",
    "Toko":     "Arunachal Pradesh",
    "Akhil":    "Telangana",
    # "Rudra":    "Rajasthan",
}

# ── Set True if farmers grew two crops in sequence ──────────
# When True, Cell 6B will ask you to pick two crop sheets and
# combine them into a single crop-system row per farmer.
IS_CROP_SYSTEM = False         # Change to True for intercrop / relay systems

# ── Root paths (edit only if your Drive layout changes) ──────
BASE_PATH   = Path("/content/drive/My Drive/MTP/Dataset/")
OUTPUT_ROOT = Path("/content/drive/My Drive/MTP/Analysis Results/")

# ── Per-cell output sub-folders (created automatically) ──────
OUT_ENERGY  = OUTPUT_ROOT / "01_Energy"
# Add any other required files according to the analysis to be done.......................................

for _p in [OUT_ENERGY]:
    _p.mkdir(parents=True, exist_ok=True)

FILE_NAME = FILE_PATHS[PERSON_NAME]
FILE_PATH = BASE_PATH/FILE_NAME
STATE     = STATE_MAP[PERSON_NAME]

# This variable tracks whether the crop-system combiner has run.
# Cell 7 onwards reads from energy_output_file, which is set in
# Cell 6 (single crop) or Cell 6B (crop system).
energy_output_file = None
system_name        = None      # e.g. "wheat_mustard"; None for single crops

print(f"Person  : {PERSON_NAME}")
print(f"State   : {STATE}")
print(f"Mode    : {'CROP SYSTEM' if IS_CROP_SYSTEM else 'SINGLE CROP'}")


Person  : Akhil
State   : Telangana
Mode    : SINGLE CROP


# ============================================================
# CELL 3 — ENERGY CONVERSION COEFFICIENTS
# ============================================================
# Sources: Pimentel (1980); Kitani (1999); Rajput et al. (2022)

In [ ]:
ENERGY_COEFF = {
    "human":                  1.96,     # MJ / man-hour
    "animal":                14.05,     # MJ / animal-hour
    "machinery":             62.70,     # MJ / tractor-hour  (embodied + fuel overhead)
    "diesel":                56.31,     # MJ / litre
    "electricity":           11.93,     # MJ / kWh
    "manure":                  0.3,     # MJ / kg manure
    "N":                     66.14,     # MJ / kg nitrogen
    "P":                     12.44,     # MJ / kg phosphate
    "K":                     11.15,     # MJ / kg potassium
    "sulphur":              120.00,    # MJ / kg micronutrient
    "zinc":                 120.00,    # MJ / kg micronutrient
    "biozinc":              120.00,    # MJ / kg micronutrient
    "potash":               120.00,    # MJ / kg micronutrient
    "boron":                120.00,    # MJ / kg micronutrient
    "monozinc":             120.00,    # MJ / kg micronutrient
    "calcium":              120.00,    # MJ / kg micronutrient
    "iron":                 120.00,    # MJ / kg micronutrient
    "ammonium sulphate":       4.3,    # MJ / kg
    "dum3":                   2.98,    # MJ / kg
    "jaivik khad":            2.98,    # MJ / kg
    "organic fertilizer":     2.98,    # MJ / kg
    "irrigation":             1.02,    # MJ / m³ water
    "herbicide":            288.00,    # MJ / kg a.i.
    "insecticide":          101.20,    # MJ / kg a.i.
    "fungicide":            216.00,    # MJ / kg a.i.
} # Add anything left micronutrients, and super fast.....

# # GHG: only inputs with direct fossil-fuel or synthetic-chemical origin
# GHG_COEFF = {
#     "machinery":   0.071,    # kg CO₂-eq / tractor-hour
#     "diesel":      2.760,    # kg CO₂-eq / litre
#     "electricity": 0.608,    # kg CO₂-eq / kWh  (Indian grid emission factor)
#     "N":           1.300,    # kg CO₂-eq / kg nitrogen (N₂O + manufacturing)
#     "P":           0.200,    # kg CO₂-eq / kg phosphate
#     "K":           0.200,    # kg CO₂-eq / kg potassium
#     "herbicide":   5.100,    # kg CO₂-eq / kg a.i.
#     "insecticide": 6.300,    # kg CO₂-eq / kg a.i.
#     "fungicide":   3.900,    # kg CO₂-eq / kg a.i.
# }

# Embodied energy of seed (MJ / kg) — crop-specific
SEED_COEFF = {
    "cotton":       25.0,
    "soyabean":      3.6,
    "kabuli_chana": 14.7,
    "desi_chana":   14.7,
    "corn":         14.7,
    "maize":        14.7,
    "wheat":        20.1,
    "chilli":       1.17,
    "mustard":      25.0,
    "jowar":        14.7,
    "bajra":        14.7,
    "seasame":      25.0,
    "gram":         14.7,
    "sugarcane":     5.3,
    "paddy":        14.7,
    "potato":       3.6,
    "arhar":        14.7,
    # "ash gourd":       ,  # NA
    "barley":       14.7,
    # "bitter guard":    ,  # NA
    "brinjal":      0.28,
    "cabbage":      2.36,
    "cardamom":     20.26,
    "carrot":       79.45,
    "cauliflower":   1.2,
    # "ginger":          ,  # NA
    "guava":        2.84,
    # "ivy guard":       ,  # NA
    "jundi":        12.6,  #sorghum
    "lady finger":  25.6,
    "mango":         2.3,
    # "marigold":        ,  # NA
    "millet":       14.7,
    "moong":        14.7,
    "onion":        14.7,
    "orange":       14.7,
    "peanut":       25.0,
    "peas":          1.9,
    "pulses":       14.7,
    "pumpkin":       1.9,
    "rose":          4.2,
    # "turmeric":        ,
    "urad":         14.7,
    # "vegetable":       ,  # NA
}

# Output energy conversion factor (MJ / kg grain or produce)
OUTPUT_COEFF = {
    "cotton":       11.8,
    "soyabean":     25.0,
    "kabuli_chana": 14.7,
    "desi_chana":   14.7,
    "corn":         14.7,
    "maize":        14.7,
    "wheat":        14.48,
    "chilli":       12.24,
    "mustard":      25.0,
    "jowar":        14.7,
    "bajra":        14.7,
    "seasame":      25.0,
    "gram":         14.7,
    "sugarcane":    16.0,
    "paddy":        14.7,
    "potato":       3.6,
    "arhar":        14.7,
    # "ash gourd":       ,
    "barley":       14.7,
    # "bitter guard":    ,
    "brinjal":      0.8,
    "cabbage":      1.2,
    "cardamom":     20.26,
    "carrot":       1.96,
    "cauliflower":   1.2,
    # "ginger":          ,
    "guava":        2.84,
    # "ivy guard":       ,
    "jundi":        12.6,  #sorghum
    "lady finger":  25.6,
    "mango":         2.3,
    # "marigold":        ,
    "millet":       14.7,
    "moong":        14.7,
    "onion":         1.6,
    "orange":        1.9,
    "peanut":       25.0,
    "peas":          1.9,
    "pulses":       14.7,
    "pumpkin":       0.8,
    "rose":          4.2,
    # "turmeric":        ,
    "urad":         14.7,
    # "vegetable":       ,
}

# Fertiliser nutrient fractions (kg nutrient / kg fertiliser product)
FERT_MAP = {
    "urea":                     {"N": 0.46},
    "dap":                      {"N": 0.18, "P": 0.46},
    "mop":                      {"K": 0.60},
    "potash":                   {"K": 0.60},
    "ssp":                      {"P": 0.16},
    "single super phosphate":   {"P": 0.16},
    "super fast":               {"P": 0.16},
    "mpp(0-52-34)":             {"P": 0.52, "K": 0.34},
    "npk(12-32-16)":            {"N": 0.12, "P": 0.32, "K": 0.16},
}

HP_TO_KW = 0.746    # conversion constant

print("Coefficients loaded.")

Coefficients loaded.


# ============================================================
# CELL 4 — UTILITY FUNCTIONS
# ============================================================

In [ ]:
def extract_number(value):
    """Return the first numeric value found in a cell; 0.0 if none."""
    if pd.isna(value):
        return 0.0
    matches = re.findall(r"[-+]?\d*\.?\d+", str(value))
    return float(matches[0]) if matches else 0.0


def parse_tractor_animal(value):
    """
    Parse a mixed tractor/animal hours cell, e.g. '4 + 2 animal'.
    Returns (tractor_hrs, animal_hrs).
    The cell text is lowercased before parsing to handle varied casing.
    """
    text = str(value).lower()
    tractor_hrs, animal_hrs = 0.0, 0.0
    for part in text.split("+"):
        hrs = extract_number(part)
        if "animal" in part:
            animal_hrs += hrs
        else:
            tractor_hrs += hrs
    return tractor_hrs, animal_hrs

def safe_division(numerator, denominator):
    """Performs division, returning np.nan if denominator is zero."""
    return numerator / denominator if denominator > 0 else np.nan

def calculate_fertilizer_energy_ghg(names_raw, amounts_raw):
    """
    Compute fertiliser energy contributions for N, P, K, and direct micronutrients.
    names_raw  : comma- or slash-separated fertiliser names (case-insensitive).
    amounts_raw: corresponding amounts in kg, same separator as names.
    Returns a dict with keys e_N, e_P, e_K, e_micronutrients_direct.
    """
    e_N = e_P = e_K = 0.0
    e_micronutrients_direct = 0.0 # Energy from micronutrients like sulphur, zinc, etc.

    names   = re.split(r"[,/]", str(names_raw).lower())
    amounts = re.split(r"[,/]", str(amounts_raw).lower())

    # List of micronutrients to track explicitly if not already handled by FERT_MAP for NPK content.
    # 'potash' is covered by FERT_MAP for K, so its energy is accounted for under e_K.
    EXPLICIT_MICRONUTRIENT_NAMES = [
        "sulphur", "zinc", "biozinc", "monozinc", "boron", "calcium", "iron"
    ]

    for i, name in enumerate(names):
        name = name.strip()
        amt = extract_number(amounts[i]) if i < len(amounts) else 0.0

        # Prioritize NPK fertilizers from FERT_MAP
        nutrient_map = FERT_MAP.get(name)
        if nutrient_map:
            for nutrient, fraction in nutrient_map.items():
                kg = amt * fraction
                e_val = kg * ENERGY_COEFF[nutrient]
                if nutrient == "N":
                    e_N += e_val
                elif nutrient == "P":
                    e_P += e_val
                elif nutrient == "K":
                    e_K += e_val
        # If not an NPK fertilizer from FERT_MAP, check if it's a direct micronutrient
        elif name in EXPLICIT_MICRONUTRIENT_NAMES:
            if name in ENERGY_COEFF:
                e_micronutrients_direct += amt * ENERGY_COEFF[name]
            else:
                warnings.warn(f"Micronutrient '{name}' found in input but no energy coefficient in ENERGY_COEFF.")

    return {"e_N": e_N, "e_P": e_P, "e_K": e_K, "e_micronutrients_direct": e_micronutrients_direct}


print("Utility functions defined.")

Utility functions defined.


# ============================================================
# CELL 5 — LOAD & RESHAPE SURVEY DATA
# ============================================================

In [ ]:
df_params = pd.read_excel(FILE_PATH, sheet_name="Energy_Params")
print(f"Energy params loaded: {len(df_params)} parameters")

# Reshape: rows=parameters → columns=parameters, index=farmers
df = df_params.set_index("name").T.reset_index()
df.rename(columns={"index": "Farmer_Col"}, inplace=True)
print(f"Reshaped: {len(df)} farmer rows × {len(df.columns)} columns")
print(df.head())

Energy params loaded: 18 parameters
Reshaped: 299 farmer rows × 19 columns
name    Farmer_Col           survey_id          crop area_ha  \
0       HANUMANTHU        MAN_062025_1          Rice    0.95   
1     HANUMANTHU.1        MAN_062025_1          Corn    0.63   
2     HANUMANTHU.2        MAN_062025_1        Cotton    0.61   
3        RAVINDHAR        MAN_062025_2  Sesame Seeds    6.25   
4      RAVINDHAR.1  MAN_062025_21QWASZ          Rice    0.32   

name irrigation_water_m3 pump_rating pump_hrs seed_kg tractor_hrs  \
0                    NaN         7.5      285      80           6   
1                    NaN         7.5       79      16           2   
2                    NaN         7.5      106      11           4   
3                    NaN         7.5      704      28          16   
4                    NaN         7.5       96      40           3   

name fuel_consumption_ltr/hrs total_fuel_consumption total_man_days  \
0                         3.5                    NaN  

# ============================================================
# CELL 6 — ENERGY CALCULATION (SINGLE CROP)
# ============================================================

In [ ]:
results = []

for _, row in df.iterrows():
    crop = str(row.get("crop", "unknown")).lower().strip()
    area = extract_number(row.get("area_ha", 0))
    if area <= 0:
        continue

    # ── Machinery & fuel ────────────────────────────────────
    t_hrs, a_hrs = parse_tractor_animal(row.get("tractor_hrs", 0))
    total_diesel = extract_number(row.get("total_fuel_consumption", 0));

    # ── Electricity (pump irrigation) ────────────────────────
    pump_hrs_val   = extract_number(row.get("pump_hrs", 0))
    pump_rating_kw = extract_number(row.get("pump_rating", 0)) * HP_TO_KW
    # elec_kwh       = 0.0 # Initialize to 0

    # Check if the engine type is electric before calculating elec_kwh
    engine_type_raw = str(row.get("engine_type", "")).lower()
    if "electric" in engine_type_raw:
        elec_kwh = pump_hrs_val * pump_rating_kw
    else:
        elec_kwh = 0.0

    #at some places where it is pipe dia in place of pump rating there params are 0 for pump rating, water and pump hours (eg: LY coulmn)

    # print(f"Survey ID: {row['survey_id']}, Farmer: {row['Farmer_Col']}, Crop: {row.get('crop', 'Unknown')}, Engine Type: {engine_type_raw}, Pump Rating: {row.get('pump_rating', 0)}, Pump Hours: {pump_hrs_val}, Elec_kwh: {elec_kwh}")

    # ── Energy inputs (MJ, farm total before per-ha conversion) ─
    e_human  = extract_number(row.get("total_man_days", 0)) * 8 * ENERGY_COEFF["human"]
    e_animal = a_hrs        * ENERGY_COEFF["animal"]
    e_mach   = t_hrs        * ENERGY_COEFF["machinery"]
    e_diesel = total_diesel * ENERGY_COEFF["diesel"]
    e_elec   = elec_kwh     * ENERGY_COEFF["electricity"]
    e_irr    = extract_number(row.get("irrigation_water_m3", 0)) * ENERGY_COEFF["irrigation"]
    e_seed   = extract_number(row.get("seed_kg", 0)) * SEED_COEFF.get(crop, 14.7)
    # Pesticides: survey records in ml (a.i.); divide by 1000 → kg a.i.
    e_herb   = extract_number(row.get("herbicides_ml",   0)) / 1000 * ENERGY_COEFF["herbicide"]
    e_insect = extract_number(row.get("insecticides_ml", 0)) / 1000 * ENERGY_COEFF["insecticide"]
    e_fung   = extract_number(row.get("fungicides_ml",   0)) / 1000 * ENERGY_COEFF["fungicide"]

    fert = calculate_fertilizer_energy_ghg(
        row.get("fert_names", ""), row.get("fert_amount_kg", ""))
    e_N, e_P, e_K = fert["e_N"], fert["e_P"], fert["e_K"]
    e_micronutrients = fert["e_micronutrients_direct"]
    #g_N, g_P, g_K = fert["g_N"], fert["g_P"], fert["g_K"]

    # # ── GHG inputs (kg CO₂-eq, farm total) ───────────────────
    # # Human, animal, irrigation and seed have no direct fossil GHG; excluded.
    # g_mach   = t_hrs        * GHG_COEFF["machinery"]
    # g_diesel = total_diesel * GHG_COEFF["diesel"]
    # g_elec   = elec_kwh     * GHG_COEFF["electricity"]
    # g_herb   = extract_number(row.get("herbicides_ml",   0)) / 1000 * GHG_COEFF["herbicide"]
    # g_insect = extract_number(row.get("insecticides_ml", 0)) / 1000 * GHG_COEFF["insecticide"]
    # g_fung   = extract_number(row.get("fungicides_ml",   0)) / 1000 * GHG_COEFF["fungicide"]

    # ── Totals ────────────────────────────────────────────────
    total_e_in  = (e_human + e_animal + e_mach + e_diesel + e_elec
                   + e_irr + e_seed + e_N + e_P + e_K + e_micronutrients
                   + e_herb + e_insect + e_fung)
    # total_g_in  = (g_mach + g_diesel + g_elec
    #                + g_N + g_P + g_K
    #                + g_herb + g_insect + g_fung)
    total_e_out = extract_number(row.get("yields_kg", 0)) * OUTPUT_COEFF.get(crop, 14.7)

    results.append({
        "Survey_ID": row["survey_id"],
        "Crop":      row.get("crop", "Unknown"),
        "State":     STATE,
        "Farmer":    row["Farmer_Col"],
        "Area_ha":   area,
        # ── Energy breakdown (MJ/ha) ──
        "Human_Energy_MJ_per_ha":        safe_division(e_human,  area),
        "Animal_Energy_MJ_per_ha":       safe_division(e_animal, area),
        "Machinery_Energy_MJ_per_ha":    safe_division(e_mach,   area),
        "Diesel_Energy_MJ_per_ha":       safe_division(e_diesel, area),
        "Electricity_Energy_MJ_per_ha":  safe_division(e_elec,   area),
        "Irrigation_Energy_MJ_per_ha":   safe_division(e_irr,    area),
        "Seed_Energy_MJ_per_ha":         safe_division(e_seed,   area),
        "N_Energy_MJ_per_ha":            safe_division(e_N,      area),
        "P_Energy_MJ_per_ha":            safe_division(e_P,      area),
        "K_Energy_MJ_per_ha":            safe_division(e_K,      area),
        "Herbicide_Energy_MJ_per_ha":    safe_division(e_herb,   area),
        "Insecticide_Energy_MJ_per_ha":  safe_division(e_insect, area),
        "Fungicide_Energy_MJ_per_ha":    safe_division(e_fung,   area),
        "Micronutrients_Energy_MJ_per_ha": safe_division(e_micronutrients, area),
        # # ── GHG breakdown (kg CO₂-eq/ha) ──
        # "Machinery_GHG_kgCO2_per_ha":    g_mach   / area,
        # "Diesel_GHG_kgCO2_per_ha":       g_diesel / area,
        # "Electricity_GHG_kgCO2_per_ha":  g_elec   / area,
        # "N_GHG_kgCO2_per_ha":            g_N      / area,
        # "P_GHG_kgCO2_per_ha":            g_P      / area,
        # "K_GHG_kgCO2_per_ha":            g_K      / area,
        # "Herbicide_GHG_kgCO2_per_ha":    g_herb   / area,
        # "Insecticide_GHG_kgCO2_per_ha":  g_insect / area,
        # "Fungicide_GHG_kgCO2_per_ha":    g_fung   / area,
        # ── Summary indices ──
        "Input_Energy_MJ_per_ha":        safe_division(total_e_in,  area),
        "Output_Energy_MJ_per_ha":       safe_division(total_e_out, area),
        "Net_Energy_MJ_per_ha":          safe_division((total_e_out - total_e_in), area),
        "EUE":  safe_division(total_e_out, total_e_in),
        "Specific_Energy_MJ_per_Kg":     safe_division(total_e_in, extract_number(row.get("yields_kg", 0))),
        "Human_EUE":        safe_division(total_e_out, e_human),
        "Animal_EUE":       safe_division(total_e_out, e_animal),
        "Machinery_EUE":    safe_division(total_e_out, e_mach),
        "Diesel_EUE":       safe_division(total_e_out, e_diesel),
        "Electricity_EUE":  safe_division(total_e_out, e_elec),
        "Irrigation_EUE":   safe_division(total_e_out, e_irr),
        "Seed_EUE":         safe_division(total_e_out, e_seed),
        "N_EUE":            safe_division(total_e_out, e_N),
        "P_EUE":            safe_division(total_e_out, e_P),
        "K_EUE":            safe_division(total_e_out, e_K),
        "Herbicide_EUE":    safe_division(total_e_out, e_herb),
        "Insecticide_EUE":  safe_division(total_e_out, e_insect),
        "Fungicide_EUE":    safe_division(e_fung,   area),
        "Micronutrients_EUE": safe_division(total_e_out, e_micronutrients)
        # "GHG_kgCO2_per_ha":              total_g_in  / area,
    })

res_df = pd.DataFrame(results)

# Save: one sheet per crop → 01_Energy_GHG folder
energy_output_file = OUT_ENERGY / f"Energy_{PERSON_NAME}.xlsx"
with pd.ExcelWriter(energy_output_file, engine="openpyxl") as writer:
    for crop in sorted(res_df["Crop"].unique()):
        crop_df = res_df[res_df["Crop"] == crop].copy()
        crop_df.insert(0, "ID", range(1, len(crop_df) + 1))
        crop_df.to_excel(writer, sheet_name=str(crop)[:31], index=False)

print(f"Energy analysis complete.")
print(f"  Farmers processed : {len(res_df)}")
print(f"  Crops found       : {sorted(res_df['Crop'].unique())}")
print(f"  Output saved to   : {energy_output_file}")

Energy analysis complete.
  Farmers processed : 278
  Crops found       : ['Chilli', 'Corn', 'Cotton', 'Pulses', 'Rice', 'Sesame Seeds']
  Output saved to   : /content/drive/My Drive/MTP/Analysis Results/01_Energy/Energy_Akhil.xlsx
